In [1]:
from pathlib import Path
from time import perf_counter
from IPython.display import display, HTML

from qdk import Context, qsharp
from qdk.qre import SequentialCountingProfiler, TemporalProfiler, TemporalProfilerConfig, estimate
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode, TwoDimensionalYokedSurfaceCode
from qdk.qre.application import QSharpApplication
from qdk.qre.profiler import QREProfiler

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "library/arithmetic/qsharp.json").is_file()
)
QIR_DIR = REPO_ROOT / "source/qep/tmp"
PROFILE_DIR = REPO_ROOT / "source/qep/profile_viewer/profiles"

# Future hardware target: physical and raw-T errors 1e-4, code cycles 1 us.
# Uniform Clifford latency conservatively allows three logical time steps.
def config(record_trace=False, aggregation_window_sec=1.0, t_factories_count=10, ecc_distance=9):
    return TemporalProfilerConfig(
        clifford_gate_time_nanos=3 * ecc_distance * 1000,
        measurment_time_nanos=1000,
        seed=42,
        record_trace=record_trace,
        aggregation_window_nanos = int(aggregation_window_sec*1e9),
        t_factories_count=t_factories_count,
        ecc_distance=ecc_distance,
    )

# Displays and saves the measured resource estimates and additive fault budget.
def report_result(result, name, t0=None):
    if t0 is not None:
        dt = perf_counter() - t0
        count = result.totals.total_operations_count
        ns_per_op = int(1e9 * dt / count)    
        print(f"Profiling time {dt:.2f}s, {ns_per_op} ns/op")

    print(f"Estimated runtime: {result.totals.total_runtime_sec}s")
    print("Logical qubits:", result.totals.logical_qubits)
    print("  of them, logical compute qubits:", result.totals.logical_compute_qubits)
    print("Physical qubits:", result.totals.physical_qubits)
    print("Estimated fault budget (not exact probability):", result.totals.error)
    
    json_path = PROFILE_DIR / (name + ".json")
    result.to_json(json_path)
    print(f"Profile saved to {json_path}")
    display(HTML(f'<a href="http://localhost:3000/source/qep/profile_viewer/?profile_name={name}">View profile</a>'))

In [2]:
qir_path = str(QIR_DIR / "shors_16.ll")
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")

QREProfiler.profile_qir(qir_path, profiler)

ProfilerResult { totals: TotalCounts { logical_qubits: 391, logical_compute_qubits: 81, physical_qubits: 62951, total_runtime_sec: 45.859711, total_operations_count: 4675584, calls_per_operation: {}, runtime_sec_per_operation: {}, error: 0.0007951529200666831 }, time_series_count: 14, aggregation_window_sec: Some(1.0), infos: [] }

## Example 1 - CCNOT gate

This example has one CCNOT gate that is transpiled into Clifford+T gates (using 7 T or T-adjoint gates).
Then we use T factories to generate T states for T/T-adjoint gates.

We run porfiler with 1 factory and see how waiting for next T state becomes a bottleneck.

Then we run profiler with 5 factories and see how states form different factories are used and T gate is executed faster.

In [3]:
qsharp.init(target_profile=qsharp.TargetProfile.Base)
app_example1 = QSharpApplication("""{
    use qs = Qubit[3];
    H(qs[0]);
    H(qs[1]);
    CCNOT(qs[0], qs[1], qs[2]);
    [MResetZ(qs[0]), MResetZ(qs[1]), MResetZ(qs[2])]  
}""")

profiler = TemporalProfiler(config(record_trace=True,t_factories_count=1))
profiler.add_adapter("transpiler")
result = app_example1.profile(profiler, use_qir=True)
report_result(result, f"example1_1factory")

Estimated runtime: 0.001108s
Logical qubits: 34
  of them, logical compute qubits: 3
Physical qubits: 5474
Estimated fault budget (not exact probability): 1.6294333333333337e-09
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/example1_1factory.json


In [4]:
profiler = TemporalProfiler(config(record_trace=True,t_factories_count=5))
profiler.add_adapter("transpiler")
result = app_example1.profile(profiler, use_qir=True)
report_result(result, f"example1_5factories")

Estimated runtime: 0.000457s
Logical qubits: 158
  of them, logical compute qubits: 3
Physical qubits: 25438
Estimated fault budget (not exact probability): 1.2491000000000004e-09
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/example1_5factories.json


## Example 2 - Grover search (from Q# samples)

In [5]:
ctx1=Context(target_profile=qsharp.TargetProfile.Base)
with open(REPO_ROOT / "samples/algorithms/Grover.qs") as f:
    ctx1.eval(f.read())
ctx1.eval("""
    operation GroverSearchEntryPoint(nQubits: Int) : Result[] {
        let nIterations = IterationsToMarked(nQubits);
        GroverSearch(nQubits, nIterations, ReflectAboutMarked)
    }
""")
 
n=6
app_grover = QSharpApplication(ctx1.code.GroverSearchEntryPoint, args=(n,))

t0 = perf_counter()
profiler = TemporalProfiler(config(record_trace=True, aggregation_window_sec=0.1))
profiler.add_adapter("transpiler")
result = app_grover.profile(profiler, use_qir=True)
report_result(result, f"grover_{n}", t0=t0)

Profiling time 0.06s, 15053 ns/op
Estimated runtime: 0.023154s
Logical qubits: 321
  of them, logical compute qubits: 11
Physical qubits: 51681
Estimated fault budget (not exact probability): 7.161866666666668e-08
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/grover_6.json


In [6]:
n=16
app_grover = QSharpApplication(ctx1.code.GroverSearchEntryPoint, args=(n,))

t0 = perf_counter()
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
result = app_grover.profile(profiler, use_qir=True)
report_result(result, f"grover_{n}", t0=t0)

Profiling time 0.64s, 1502 ns/op
Estimated runtime: 1.594647s
Logical qubits: 341
  of them, logical compute qubits: 31
Physical qubits: 54901
Estimated fault budget (not exact probability): 1.0035700833333334e-05
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/grover_16.json


## Example 3 - modular multiplication

In [7]:
# ModMul search example.
ctx2=Context(project_root=str(REPO_ROOT / "library/arithmetic"),
            target_profile=qsharp.TargetProfile.Adaptive)
ctx2.eval("""
    operation ProfileModMul(n: Int) : Unit {
        let modulus = (1L <<< n) - 1L;
        use xs = Qubit[n];
        use ys = Qubit[n];
        use ans = Qubit[n];
        ModMul.ModMul(xs, ys, ans, modulus);
    }
""")

n=8
app_modmul = QSharpApplication(ctx2.code.ProfileModMul, args=(n,))

profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = app_modmul.profile(profiler, use_qir=True)
report_result(result, f"modmul_{n}", t0=t0)

Profiling time 0.19s, 8545 ns/op
Estimated runtime: 0.244396s
Logical qubits: 343
  of them, logical compute qubits: 33
Physical qubits: 55223
Estimated fault budget (not exact probability): 9.777535999999996e-07
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/modmul_8.json


In [8]:
n=200
app_modmul = QSharpApplication(ctx2.code.ProfileModMul, args=(n,))

profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = app_modmul.profile(profiler, use_qir=True)
report_result(result, f"modmul_{n}", t0=t0)

Profiling time 1.76s, 107 ns/op
Estimated runtime: 178.695958s
Logical qubits: 1111
  of them, logical compute qubits: 801
Physical qubits: 178871
Estimated fault budget (not exact probability): 0.012165281140733565
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/modmul_200.json


## Example 4 - Shor's algorithm (16 bit and 50 bit) from precompiled QIR file

These are small, single-shot Shor workloads, not RSA-2048 estimates. We target future hardware with physical and raw T-state errors of $10^{-4}$, a $1\,\mu s$ syndrome cycle, and distance 9. A uniform Clifford latency of three logical steps is a conservative simplification; measurement and reset each take one physical cycle.

Each single-level 15-to-1 factory uses the 31-patch, 11-logical-step RM-preparation model from [arXiv:2211.07629, Appendix C](https://arxiv.org/abs/2211.07629). Factories and compute memory share its distance-dependent logical error model. Cached T states also accrue memory error until injection finishes.

The reported error is a first-order additive fault budget, not an exact probability of an incorrect factor. Contributions are charged once, rather than charging the same history on every measurement. This simplified model does not simulate error propagation, routing, decoder delays, synthesis error, or the algorithm's intrinsic success probability. The examples below assert a fault budget below 0.33.

In [9]:
qir_path = str(QIR_DIR / "shors_16.ll")
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = QREProfiler.profile_qir(qir_path, profiler)
assert result.totals.error < 0.33
report_result(result, f"shors_16", t0=t0)

Profiling time 0.66s, 140 ns/op
Estimated runtime: 45.859711s
Logical qubits: 391
  of them, logical compute qubits: 81
Physical qubits: 62951
Estimated fault budget (not exact probability): 0.0007951529200666831
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/shors_16.json


In [10]:
qir_path = str(QIR_DIR / "shors_50.ll")
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = QREProfiler.profile_qir(qir_path, profiler)
assert result.totals.error < 0.33
report_result(result, f"shors_50", t0=t0)

Profiling time 14.89s, 109 ns/op
Estimated runtime: 1457.791958s
Logical qubits: 561
  of them, logical compute qubits: 251
Physical qubits: 90321
Estimated fault budget (not exact probability): 0.07499182121755778
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/shors_50.json


The same algorithm, but using 4 T-factories instead of 10.

In [11]:
qir_path = str(QIR_DIR / "shors_50.ll")
profiler = TemporalProfiler(config(t_factories_count=4))
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = QREProfiler.profile_qir(qir_path, profiler)
assert result.totals.error < 0.33
report_result(result, f"shors_50_4factories", t0=t0)

Profiling time 14.52s, 106 ns/op
Estimated runtime: 1588.21793s
Logical qubits: 375
  of them, logical compute qubits: 251
Physical qubits: 60375
Estimated fault budget (not exact probability): 0.07999342641753275
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/shors_50_4factories.json


## Example 5 - VQE energy evaluation

Profile one VQE energy-evaluation workload adapted from
[Variational Quantum Algorithms](../../../samples/chemistry/Variational%20Quantum%20Algorithms.ipynb).
This uses the sample's seven-term, two-qubit Hamiltonian and Pauli-rotation ansatz,
with 100 shots per term. Angles are chosen as multiples of pi/8 so the default
transpiler can synthesize them. The profiler models execution costs with random
measurement results; it does not compute a physically meaningful energy or run
the classical optimization loop.

In [12]:
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive)
app_vqe = QSharpApplication("""
{
    let terms = [
        (-3.0, [PauliI, PauliZ]),
        (0.5, [PauliX, PauliI]),
        (1.5, [PauliX, PauliX]),
        (-0.5, [PauliX, PauliZ]),
        (-1.5, [PauliY, PauliY]),
        (2.5, [PauliZ, PauliI]),
        (-1.5, [PauliZ, PauliZ])
    ];
    mutable energy = 0.0;
    for (coefficient, bases) in terms {
        mutable total = 0.0;
        for _ in 1..100 {
            use register = Qubit[2];
            // Exp(ZZ, pi/8) and Exp(XY, -pi/8), decomposed for this backend.
            CNOT(register[0], register[1]);
            Rz(-Std.Math.PI() / 4.0, register[1]);
            CNOT(register[0], register[1]);
            within {
                H(register[0]);
                Adjoint S(register[1]);
                H(register[1]);
            } apply {
                CNOT(register[0], register[1]);
                Rz(Std.Math.PI() / 4.0, register[1]);
                CNOT(register[0], register[1]);
            }
            let measured = Measure(bases, register);
            if measured == Zero {
                total += 1.0;
            } else {
                total -= 1.0;
            }
            ResetAll(register);
        }
        energy += coefficient * total / 100.0;
    }
    energy
}
""")

profiler = TemporalProfiler(config(record_trace=True))
profiler.add_adapter("transpiler")
profiler.add_adapter("result_adapter")
t0 = perf_counter()
result = app_vqe.profile(profiler, use_qir=True)
report_result(result, "vqe", t0=t0)

Profiling time 0.10s, 5067 ns/op
Estimated runtime: 0.275571s
Logical qubits: 313
  of them, logical compute qubits: 3
Physical qubits: 50393
Estimated fault budget (not exact probability): 9.16239999999995e-08
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/vqe.json
